# 02 — Pipeline check

**Goal:** confirm that the Milestone 2 pipeline produced what we intended before training on it in Milestone 3.

The pipeline (`python -m deadlock_predictor.data.pipeline`) turns raw `match_player` rows into `data/processed/snapshots.parquet`: **one row per (match, snapshot time)** holding the game state at time *t* from Team0's point of view, the label `team0_win`, and a `split` column. Here we check:
- how many matches each filter removed;
- that the train/validation/test split is clean and chronological;
- that there is no leakage left;
- what the features look like and which ones carry signal.

**From section 4 on we look only at the training split.** Every choice we make after looking at data is a little bit of fitting. If we peeked at the test set while choosing features, the test score would no longer be an honest estimate of performance on new matches.

**Before running:** select the `Python (deadlock)` kernel and run the pipeline once (~15 min the first time, mostly downloading).

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deadlock_predictor.features.snapshots import FEATURES, LABEL

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
assert (PROCESSED / "snapshots.parquet").exists(), (
    "Run `python -m deadlock_predictor.data.pipeline` first"
)

snaps = pd.read_parquet(PROCESSED / "snapshots.parquet")
summary = json.loads((PROCESSED / "summary.json").read_text())
plt.rcParams["figure.figsize"] = (9, 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", message="The inplace keyword")
print(f"{len(snaps):,} rows, {snaps.match_id.nunique():,} matches, {snaps.shape[1]} columns")
snaps.head()

## 1. What the filters removed

A match is kept only if it has exactly 6 players per team, nobody abandoned, it was scored, and it has a winner. One match can fail several checks, so the removals can add up to more than the difference.

In [ ]:
f = summary["filters"]
removed = pd.Series(
    {k: v for k, v in f.items() if k not in ("matches", "kept")}, name="matches failing"
)
since = summary["start_time"][:10]
kept = f"kept {f['kept']:,} ({f['kept'] / f['matches']:.1%})"
print(f"{f['matches']:,} ranked 6v6 matches since {since}, {kept}")
removed.to_frame().assign(share=lambda d: (d["matches failing"] / f["matches"]).round(4))

Abandons are by far the biggest filter. Dropping them is a V1 simplification: the model never sees a 6v5 game, so it would be overconfident about one. That's acceptable for historical curves of normal matches.

## 2. The split

Matches are ordered by start time: the oldest 70% are **train**, the next 15% **validation** (for choosing models and settings), the newest 15% **test** (touched once, at the end). All of a match's snapshots share its start time, so a match can never be in two splits.

In [ ]:
per_match = snaps.drop_duplicates("match_id").set_index("match_id")
assert snaps.groupby("match_id").split.nunique().max() == 1, "a match is in two splits"

table = per_match.groupby("split").agg(
    matches=(LABEL, "size"),
    first_start=("start_time", "min"),
    last_start=("start_time", "max"),
    team0_win_rate=(LABEL, "mean"),
)
table["rows"] = snaps.groupby("split").size()
table["win_rate_95ci"] = 1.96 * np.sqrt(
    table.team0_win_rate * (1 - table.team0_win_rate) / table.matches
)
table = table.loc[["train", "validation", "test"]]
assert (table.last_start.iloc[:-1].values < table.first_start.iloc[1:].values).all(), (
    "splits overlap in time"
)
table.round({"team0_win_rate": 4, "win_rate_95ci": 4})

In [ ]:
daily = per_match.groupby([per_match.start_time.dt.date, "split"]).size().unstack(fill_value=0)
ax = daily[["train", "validation", "test"]].plot.bar(stacked=True, width=1, figsize=(12, 3))
ax.set(title="Matches per day by split", xlabel="", ylabel="matches")
ax.set_xticks(ax.get_xticks()[::3]);

**Base rate.** Team0 wins 50.7% ± 0.2% of training matches: a small side advantage that is statistically real (the interval excludes 50%), but it is gone in validation and test (~50.0%). So the game itself changed a little over the patch, perhaps through a hotfix or players adapting. That's a small example of why we test on *later* matches: whatever the model learns about the past must still hold in the future. For scoring, the difference is negligible: always predicting 50% gives log loss ≈ 0.693 on every split. That's the number Milestone 3's baseline must beat.

**The data gap.** The chart shows far fewer matches on Oct 20–21 (~11k and ~3k, against ~22k on a normal day). The dump itself is missing them; the extractor read everything there was. If deadlock-api backfills those days, the rows arrive in delta files, and `python -m deadlock_predictor.data.pipeline --force` picks them up.

## 3. Leakage checks

The pipeline promises that every feature at time *t* uses only information available at *t*. We can't prove that from the table alone, but we can check the symptoms of the leaks we know about:
- no snapshot at or after the end of the match;
- objective counts only ever grow during a match (an objective can't be un-destroyed), and never exceed what exists;
- no feature is suspiciously close to the label.

In [ ]:
print("rows with t >= duration:", int((snaps.t >= snaps.duration_s).sum()))
print(f"latest snapshot as share of duration: {(snaps.t / snaps.duration_s).max():.4f}")

obj_cols = [c for c in FEATURES if c.startswith("obj_")]
own_lost_or_taken = snaps.groupby("match_id")[obj_cols].diff().dropna()
print(
    "objective diffs that decreased between snapshots:",
    int((own_lost_or_taken < 0).sum().sum()),
    "(fine: the other team can take one too)",
)
print("max |objective diff| per group:", snaps[obj_cols].abs().max().to_dict())

A decreasing objective diff is fine (Team1 destroyed one of Team0's towers), so the meaningful check is the bounds: at most 3 Tier 1, 3 Tier 2 and 3 barracks per team (one per lane), 2 shield generators, 1 Titan. The rest of the leakage protection is enforced by the unit tests in `tests/test_snapshots.py`, which run on every commit.

## 4. The features (train only)

Every `_diff` feature is Team0 minus Team1, so positive means Team0 is ahead. Using differences instead of each team's raw totals halves the number of inputs and builds in a symmetry we believe: what matters is who's ahead, not by how much each team has in absolute terms.

`t` is game time in seconds, and `badge_mean` is the lobby's average rank. (Notebook 01 assumed `average_badge` was per player; it turned out to be the lobby average copied onto every player, so there's no team rank gap to use.)

In [ ]:
train = snaps[snaps.split == "train"]
train[FEATURES + [LABEL]].describe().T.round(2)

How strongly does each feature relate to the outcome? The correlation between a feature and a 0/1 label (point-biserial correlation) is a quick, rough measure. It only captures *linear* association, and it changes with game time, so we compute it at four times.

In [ ]:
times = [540, 900, 1500, 2100]
corr = pd.DataFrame(
    {
        f"{t // 60} min": train[train.t == t][FEATURES[1:]].corrwith(
            train[train.t == t][LABEL].astype(float)
        )
        for t in times
    }
)
ax = corr.drop(index="badge_mean").plot.barh(figsize=(9, 6))
ax.axvline(0, color="grey", lw=0.8)
ax.set(title="Correlation of each feature with Team0 winning (train)", xlabel="correlation")
corr.round(3)

Things to notice:
- **Net worth and level** carry the most signal, and it grows through the game (correlation ~0.32 at 9 minutes, ~0.66 at 35). They're also strongly correlated with each other, since levels come from souls.
- **Objectives matter in order.** Tier 1 towers carry signal early (0.22 at 9 minutes) and fade later, once both teams have lost some. Tier 2, barracks, shield generators and Titan take over as the game goes on: by 35 minutes, barracks and shield generators are among the strongest features.
- **Mid-boss** diffs are empty at 9 minutes (it hasn't been killed yet, so the correlation is undefined) and strong later.
- **Deaths are negatively correlated** with Team0 winning, as expected.
- **Creep kills and denies** carry little signal.
- `badge_mean` is the same for both teams, so on its own it can't predict *which* team wins (correlation ~0). It could still matter in combination with other features; section 5 checks that.

### Redundant features

Every kill is someone else's death, so `kills_diff` (Team0 kills − Team1 kills) and `deaths_diff` (Team0 deaths − Team1 deaths) should be almost exact mirror images: correlation −0.99. Strongly correlated inputs don't hurt a model's *predictions* much, but they make a linear model's coefficients unstable and hard to interpret: the model can split the same effect between two columns in many different ways.

In [ ]:
c = train[FEATURES[1:-1]].corr()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(c, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(c)), c.columns, rotation=90)
ax.set_yticks(range(len(c)), c.columns)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature correlations (train)")
print("corr(kills_diff, deaths_diff) =", round(c.loc["kills_diff", "deaths_diff"], 3))
print("corr(net_worth_diff, level_diff) =", round(c.loc["net_worth_diff", "level_diff"], 3))

## 5. The net worth signal, with ~55× more data

The same curve as notebook 01's 5,000-match sample, rebuilt from the processed table. It should have the same shape (rising to ~35 minutes, then falling), with much tighter error bars.

In [ ]:
d = train[train.net_worth_diff != 0].copy()
d["ahead_won"] = np.where(d.net_worth_diff > 0, d[LABEL], ~d[LABEL])
by_t = d.groupby("t").ahead_won.agg(["mean", "size"])
by_t = by_t[by_t["size"] >= 500]
by_t["ci"] = 1.96 * np.sqrt(by_t["mean"] * (1 - by_t["mean"]) / by_t["size"])
ax = by_t["mean"].plot(marker="o")
ax.fill_between(by_t.index, by_t["mean"] - by_t.ci, by_t["mean"] + by_t.ci, alpha=0.3)
ax.axhline(0.5, color="grey", ls="--")
ax.set(
    title="How often the team ahead in net worth wins (train)",
    xlabel="game time (s)",
    ylabel="win rate",
)
by_t.round(3).T

### Does lobby rank change how decisive a lead is?

At 20 minutes, split matches into rank thirds and plot win rate against net worth lead. If the curves differ in steepness, rank changes how reliably leads turn into wins: an interaction a tree model could use.

In [ ]:
at = train[train.t == 1200].copy()
at["rank"] = pd.qcut(at.badge_mean, 3, labels=["low", "mid", "high"])
edge = at.net_worth_diff.abs().quantile(0.95)
at["lead_bin"] = pd.cut(at.net_worth_diff, np.linspace(-edge, edge, 13))
fig, ax = plt.subplots()
for rank, g in at.groupby("rank", observed=True):
    rate = g.groupby("lead_bin", observed=True)[LABEL].agg(["mean", "size"])
    rate = rate[rate["size"] >= 50]
    ax.plot([b.mid for b in rate.index], rate["mean"], marker=".", label=f"{rank} rank")
ax.axhline(0.5, color="grey", ls="--")
ax.set(
    title="P(Team0 wins) vs net worth lead at 20 min, by lobby rank",
    xlabel="Team0 − Team1 net worth",
    ylabel="Team0 win rate",
)
ax.legend();

**No: the three curves lie on top of each other.** A logistic fit per rank third gives almost the same slope; a 10k lead at 20 minutes wins 75–76% at every rank. So lobby rank probably adds nothing to the model, and Milestone 3 can test that directly on validation.

## 6. Ready for Milestone 3

- **Table:** `data/processed/snapshots.parquet`, one row per (match, snapshot), label `team0_win`, `split` column; `summary.json` records the filters and split dates.
- **Baseline inputs:** `FEATURES` from `deadlock_predictor.features.snapshots`. Expect to drop or combine near-duplicates (`kills_diff`/`deaths_diff`) for an interpretable logistic regression.
- **Scale:** diffs range from single digits (objectives) to tens of thousands (net worth), so features must be **standardized** before logistic regression with regularization, or the penalty treats them unequally.
- **Time:** the value of a lead changes with `t`, so the baseline needs `t` and probably lead × time interactions.
- **Lobby rank** showed no effect here; keep it only if it helps on validation.
- **Evaluation:** fit on train, choose settings on validation, report log loss, Brier score and calibration on test once.